# KB Ablation: `--use-kb` vs `--no-use-kb`

Compares the effect of augmenting the rule-based ATT&CK tagger with retrieval from the cyber-anomaly Chroma KB.

Both runs share the same parser, windowing, chains, IsolationForest and GRU. The only difference is `tag_techniques_with_kb(use_kb=...)`.

Outputs compared:
- # of windows escalated above threshold
- distinct ATT&CK technique IDs surfaced (rule vs rule+kb)
- average hits/window
- per-window technique deltas

In [4]:
import sys, os, json, shutil
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
SRC  = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.environ.setdefault('CHROMA_EMB_DEVICE', 'cpu')
os.environ.setdefault('PYTHONIOENCODING', 'utf-8')
print('ROOT:', ROOT)

ROOT: D:\ISEP\Challange-3\DualSentinel


In [5]:
import subprocess

# The pipeline needs chromadb + sentence-transformers from the `cyber-anomaly`
# conda env. We invoke its python directly so this notebook can run from any
# kernel (including the workspace .venv which lacks those deps).
PYTHON_KB = r'C:\Users\arsen\miniconda3\envs\cyber-anomaly\python.exe'

INPUT   = ROOT / 'data' / 'samples' / 'sample_lmd.csv'
DATASET = 'lmd'
THRESH  = 0.2
OUT_BASE = ROOT / 'results' / 'kb_ablation'
if OUT_BASE.exists():
    shutil.rmtree(OUT_BASE)
OUT_BASE.mkdir(parents=True)

def run(use_kb: bool, out_dir: Path) -> None:
    cmd = [
        PYTHON_KB, str(ROOT / 'src' / 'pipeline.py'),
        '--input', str(INPUT), '--dataset', DATASET,
        '--output-dir', str(out_dir),
        '--skip-detectors', '--skip-judge',
        '--threshold', str(THRESH),
        '--use-kb' if use_kb else '--no-use-kb',
    ]
    env = {**os.environ, 'PYTHONIOENCODING': 'utf-8', 'CHROMA_EMB_DEVICE': 'cpu'}
    print(f"[{'KB' if use_kb else 'NO-KB'}] running ...")
    proc = subprocess.run(
        cmd, capture_output=True, text=True,
        encoding='utf-8', errors='replace',
        env=env, cwd=str(ROOT),
    )
    out = proc.stdout or ''
    err = proc.stderr or ''
    tail = '\n'.join(out.splitlines()[-6:])
    print(tail)
    if proc.returncode != 0:
        print('STDERR tail:\n', '\n'.join(err.splitlines()[-15:]))

run(False, OUT_BASE / 'no_kb')
run(True,  OUT_BASE / 'with_kb')
print('\nDone.')

[NO-KB] running ...
├────────────────┼────────────────────────────────────────────────────────────┤
│ Windows scored │ D:\ISEP\Challange-3\DualSentinel\results\kb_ablation\no_k… │
│ Chains         │ D:\ISEP\Challange-3\DualSentinel\results\kb_ablation\no_k… │
│ Evidence packs │ D:\ISEP\Challange-3\DualSentinel\results\kb_ablation\no_k… │
│ Report         │ D:\ISEP\Challange-3\DualSentinel\results\kb_ablation\no_k… │
└────────────────┴────────────────────────────────────────────────────────────┘
[KB] running ...
├────────────────┼────────────────────────────────────────────────────────────┤
│ Windows scored │ D:\ISEP\Challange-3\DualSentinel\results\kb_ablation\with… │
│ Chains         │ D:\ISEP\Challange-3\DualSentinel\results\kb_ablation\with… │
│ Evidence packs │ D:\ISEP\Challange-3\DualSentinel\results\kb_ablation\with… │
│ Report         │ D:\ISEP\Challange-3\DualSentinel\results\kb_ablation\with… │
└────────────────┴────────────────────────────────────────────────────────────┘

Do

## Aggregate metrics

In [6]:
def load_windows(path: Path) -> list[dict]:
    return json.loads(path.read_text())

w_nokb = load_windows(OUT_BASE / 'no_kb' / 'windows_scored.json')
w_kb   = load_windows(OUT_BASE / 'with_kb' / 'windows_scored.json')

def summarise(name: str, windows: list[dict]) -> dict:
    escalated = [w for w in windows if w['detector_score'] >= THRESH]
    all_techs = set()
    rule_techs = set()
    kb_techs = set()
    hit_counts = []
    for w in windows:
        hits = w.get('attck_hits', [])
        hit_counts.append(len(hits))
        for h in hits:
            all_techs.add(h['technique'])
            if h.get('source', 'rule') == 'rule':
                rule_techs.add(h['technique'])
            else:
                kb_techs.add(h['technique'])
    return {
        'config': name,
        'windows': len(windows),
        'escalated': len(escalated),
        'distinct_techniques_total': len(all_techs),
        'distinct_techniques_rule': len(rule_techs),
        'distinct_techniques_kb_only': len(kb_techs - rule_techs),
        'avg_hits_per_window': round(sum(hit_counts) / max(len(hit_counts), 1), 2),
    }

summary = pd.DataFrame([summarise('no_kb', w_nokb), summarise('with_kb', w_kb)])
summary

,config,windows,escalated,distinct_techniques_total,distinct_techniques_rule,distinct_techniques_kb_only,avg_hits_per_window
0,no_kb,3,2,4,4,0,1.33
1,with_kb,3,3,14,4,10,5.67


## Per-window technique sets

In [7]:
def techs_per_window(windows: list[dict]) -> list[dict]:
    rows = []
    for w in windows:
        rule = sorted({h['technique'] for h in w.get('attck_hits', []) if h.get('source','rule')=='rule'})
        kb   = sorted({h['technique'] for h in w.get('attck_hits', []) if h.get('source')=='kb'})
        rows.append({
            'window_start': str(w.get('window_start',''))[:19],
            'detector_score': round(w['detector_score'], 3),
            'rule_techs': ','.join(rule) or '-',
            'kb_techs':   ','.join(kb)   or '-',
        })
    return rows

df_kb = pd.DataFrame(techs_per_window(w_kb))
df_kb

,window_start,detector_score,rule_techs,kb_techs
0,2024-03-01T10:00:00,0.2,"T1003,T1021.002,T1059.001","T1003.001,T1027,T1059.003"
1,2024-03-01T10:01:00,0.2,-,"T1016,T1049,T1055,T1070.005,T1559"
2,2024-03-01T10:02:00,0.2,T1570,"T1021.002,T1049,T1059.003,T1218,T1587.001"


## KB-only technique frequency (newly surfaced by retrieval)

In [8]:
from collections import Counter
kb_only = Counter()
for w in w_kb:
    rule_set = {h['technique'] for h in w.get('attck_hits',[]) if h.get('source','rule')=='rule'}
    for h in w.get('attck_hits', []):
        if h.get('source') == 'kb' and h['technique'] not in rule_set:
            kb_only[(h['technique'], h['name'])] += 1
rows = [{'technique': t, 'name': n, 'windows_seen': c} for (t, n), c in kb_only.most_common()]
pd.DataFrame(rows)

,technique,name,windows_seen
0,T1059.003,Windows Command Shell,2
1,T1049,System Network Connections Discovery Via Net.EXE,2
2,T1003.001,Potential Credential Dumping Activity Via LSASS,1
3,T1027,PowerShell Base64 Encoded Invoke Keyword,1
4,T1055,CobaltStrike Named Pipe Pattern Regex,1
5,T1559,Inter-Process Communication,1
6,T1070.005,Network Share Connection Removal,1
7,T1016,Suspicious Network Command,1
8,T1021.002,SMB/Windows Admin Shares,1
9,T1587.001,Potential PsExec Remote Execution,1


## Sample evidence pack (peak window, with KB)

Verifies that KB candidates and the peak chain actually appear in the rendered evidence pack served to the SLM/Judge.

In [9]:
ep = json.loads((OUT_BASE / 'with_kb' / 'evidence_packs.json').read_text(encoding='utf-8'))
if ep:
    peak = max(ep, key=lambda x: x['detector_score'])
    print(peak['evidence_pack'])
else:
    print('No high-risk windows in this sample.')

=== EVIDENCE PACK ===
Window: 2024-03-01T10:00:00+00:00 → 2024-03-01T10:01:00+00:00
Total events: 16

--- Aggregate stats ---
Unique EventIDs: 5
Unique processes: 7
Unique users: 1
Process creations (EID 1): 6
Network connections (EID 3): 7
File creations (EID 11): 1
Registry modifications (EID 13): 1
Suspicious processes detected: 10
Lateral movement ports seen: 5
PowerShell executions: 4
CMD executions: 1
Outbound unique IPs: 6
Mimikatz present: True
PsExec present: False
EventID entropy: 1.802
Process entropy: 2.524

--- Rule tagger hits (pre-computed) ---
  T1059.001 PowerShell (confidence=0.70)
  T1021.002 SMB/Windows Admin Shares (confidence=0.60)
  T1003 Credential Dumping (confidence=0.95)

--- ATT&CK KB candidates (retrieved, not confirmed) ---
  T1003.001 Potential Credential Dumping Activity Via LSASS (similarity=0.85) :: Sigma Detection Rule: Potential Credential Dumping Activity Via LSASS
Sysmon EventID: 10 (category: process_access)
Description: Detects process access req